# Gerador de Dados Sintéticos — Eventos Urbanos (Rio de Janeiro)

Trabalho Prático — Bancos de Dados II — IC/UFRJ

Este notebook gera **dados sintéticos** de eventos urbanos (acidentes, alagamentos,
incêndios, etc.) localizados em bairros reais do município do Rio de Janeiro,
seguindo exatamente o esquema de dados definido no enunciado do trabalho.

**O que este notebook faz:**

1. Permite escolher a **quantidade de registros** a gerar (Pequeno / Médio / Grande / Personalizado).
2. Gera eventos apenas dentre os **tipos de evento permitidos** no enunciado.
3. Distribui os eventos geograficamente entre **30 bairros reais** do Rio de Janeiro, com coordenadas plausíveis (latitude/longitude com variação aleatória em torno do centro de cada bairro).
4. Exporta os dados em **JSON** (array e JSON Lines, prontos para `mongoimport`) e em **CSV**.
5. Mostra estatísticas rápidas (contagem por tipo, por bairro, evolução temporal) para conferência.

> Ajuste a semente aleatória (`SEED`) se quiser reproduzir exatamente o mesmo dataset entre execuções.

## 1. Instalação de dependências (execute uma vez, se necessário)

In [ ]:
# Descomente e execute esta célula caso as bibliotecas não estejam instaladas no seu ambiente
# !pip install ipywidgets pandas matplotlib tqdm --quiet

## 2. Imports

In [ ]:
import json
import random
import uuid
from datetime import datetime, timedelta

import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import ipywidgets as widgets
from IPython.display import display, clear_output

pd.set_option("display.max_colwidth", 80)

## 3. Configuração do domínio

Aqui ficam as constantes que definem o "mundo" da simulação: os tipos de evento
permitidos (retirados literalmente do enunciado), os bairros reais do Rio de
Janeiro com suas coordenadas centrais aproximadas, os status possíveis e os
tipos de reportante.

In [ ]:
SEED = 42
random.seed(SEED)

CIDADE = "Rio de Janeiro"

# Tipos de evento permitidos pelo enunciado do trabalho (Seção 2)
TIPOS_EVENTOS = [
    "Acidente de Trânsito",
    "Alagamento",
    "Queda de Energia",
    "Incêndio",
    "Problema no Transporte Público",
    "Vazamento de Água",
    "Interdição de Via",
    "Deslizamento de Terra",
    "Tiroteio",
]

# Pesos relativos de ocorrência (tornam a distribuição mais realista;
# ajuste livremente ou troque por [1]*len(TIPOS_EVENTOS) para uniforme)
PESOS_TIPOS = [22, 16, 10, 9, 14, 8, 9, 4, 8]

STATUS_OPCOES = ["Aberto", "Em Andamento", "Resolvido", "Fechado"]
PESOS_STATUS = [35, 25, 25, 15]

# Bairros reais do Rio de Janeiro com coordenadas centrais aproximadas
# (latitude, longitude). A localização de cada evento é sorteada com um
# pequeno deslocamento aleatório em torno desse centro (~ até 1.5 km).
BAIRROS = {
    "Centro": (-22.9068, -43.1729),
    "Tijuca": (-22.9249, -43.2277),
    "Copacabana": (-22.9711, -43.1822),
    "Ipanema": (-22.9838, -43.2096),
    "Leblon": (-22.9847, -43.2244),
    "Botafogo": (-22.9519, -43.1823),
    "Flamengo": (-22.9326, -43.1758),
    "Barra da Tijuca": (-23.0045, -43.3651),
    "Recreio dos Bandeirantes": (-23.0231, -43.4653),
    "Jacarepaguá": (-22.9569, -43.3639),
    "Campo Grande": (-22.9037, -43.5613),
    "Bangu": (-22.8792, -43.4659),
    "Madureira": (-22.8730, -43.3395),
    "Penha": (-22.8390, -43.2801),
    "Ilha do Governador": (-22.8098, -43.2075),
    "Santa Cruz": (-22.9192, -43.6864),
    "Realengo": (-22.8779, -43.4351),
    "Vila Isabel": (-22.9159, -43.2453),
    "Méier": (-22.9014, -43.2799),
    "Pavuna": (-22.8104, -43.3564),
    "Rocinha": (-22.9887, -43.2453),
    "Complexo do Alemão": (-22.8600, -43.2724),
    "Maré": (-22.8611, -43.2408),
    "São Cristóvão": (-22.8985, -43.2211),
    "Lagoa": (-22.9722, -43.2050),
    "Jardim Botânico": (-22.9707, -43.2222),
    "Gávea": (-22.9767, -43.2325),
    "Cidade de Deus": (-22.9469, -43.3628),
    "Anchieta": (-22.8267, -43.3897),
    "Guaratiba": (-23.0578, -43.5967),
}

# Tipos de reportante e prefixo do identificador
REPORTANTE_TIPOS = ["Cidadao", "Sensor", "OrgaoPublico"]
PESOS_REPORTANTE = [60, 30, 10]
PREFIXO_IDENTIFICADOR = {"Cidadao": "USR", "Sensor": "SENSOR", "OrgaoPublico": "ORG"}

# Janela temporal em que os eventos serão distribuídos
DATA_FIM = datetime(2026, 6, 30, 23, 59, 59)
DATA_INICIO = DATA_FIM - timedelta(days=365)

# Templates de descrição por tipo de evento (sorteados aleatoriamente)
DESCRICOES = {
    "Acidente de Trânsito": [
        "Colisão entre dois veículos na via principal",
        "Atropelamento registrado por transeunte",
        "Engavetamento envolvendo múltiplos veículos",
        "Colisão com poste, trânsito lento no local",
    ],
    "Alagamento": [
        "Rua completamente interditada devido ao volume de chuva",
        "Acúmulo de água impede passagem de veículos",
        "Bueiro entupido causa alagamento na via",
        "Nível da água subindo próximo a residências",
    ],
    "Queda de Energia": [
        "Falta de energia elétrica reportada por moradores",
        "Transformador danificado após tempestade",
        "Interrupção no fornecimento afeta quarteirão inteiro",
    ],
    "Incêndio": [
        "Princípio de incêndio em imóvel residencial",
        "Incêndio em vegetação próximo à via",
        "Fogo em veículo estacionado, bombeiros acionados",
    ],
    "Problema no Transporte Público": [
        "Ônibus quebrado obstruindo faixa de rolamento",
        "Atraso significativo reportado por usuários",
        "Falha mecânica em composição do sistema sobre trilhos",
    ],
    "Vazamento de Água": [
        "Rompimento de tubulação causa vazamento na via",
        "Vazamento constante reportado há mais de um dia",
        "Desperdício de água por adutora danificada",
    ],
    "Interdição de Via": [
        "Via interditada para obras de manutenção",
        "Bloqueio temporário devido a evento público",
        "Interdição por risco de desabamento parcial",
    ],
    "Deslizamento de Terra": [
        "Deslizamento de encosta após fortes chuvas",
        "Risco iminente de deslizamento reportado por moradores",
        "Barreira cede parcialmente sobre a via",
    ],
    "Tiroteio": [
        "Disparos reportados por moradores da região",
        "Confronto reportado nas proximidades",
        "Ocorrência policial em andamento no local",
    ],
}

print(f"{len(TIPOS_EVENTOS)} tipos de evento | {len(BAIRROS)} bairros carregados.")

## 4. Funções de geração

In [ ]:
def _jitter_coordenada(lat, lon, raio_graus=0.014):
    '''Aplica um pequeno deslocamento aleatório à coordenada central do bairro
    (raio_graus ~0.014 equivale a aproximadamente 1.5 km).'''
    return (
        round(lat + random.uniform(-raio_graus, raio_graus), 6),
        round(lon + random.uniform(-raio_graus, raio_graus), 6),
    )


def _data_hora_aleatoria():
    delta = DATA_FIM - DATA_INICIO
    segundos_aleatorios = random.randint(0, int(delta.total_seconds()))
    return DATA_INICIO + timedelta(seconds=segundos_aleatorios)


def gerar_evento(indice: int, largura_id: int = 8) -> dict:
    '''Gera um único evento urbano sintético seguindo o esquema do enunciado.'''
    tipo = random.choices(TIPOS_EVENTOS, weights=PESOS_TIPOS, k=1)[0]
    bairro = random.choice(list(BAIRROS.keys()))
    lat_base, lon_base = BAIRROS[bairro]
    lat, lon = _jitter_coordenada(lat_base, lon_base)

    reportante_tipo = random.choices(REPORTANTE_TIPOS, weights=PESOS_REPORTANTE, k=1)[0]
    reportante_num = random.randint(1, 5000)
    identificador = f"{PREFIXO_IDENTIFICADOR[reportante_tipo]}{reportante_num:04d}"

    evento = {
        "idEvento": f"EVT{indice:0{largura_id}d}",
        "tipo": tipo,
        "descricao": random.choice(DESCRICOES[tipo]),
        "dataHora": _data_hora_aleatoria().strftime("%Y-%m-%dT%H:%M:%S"),
        "gravidade": random.randint(1, 5),
        "status": random.choices(STATUS_OPCOES, weights=PESOS_STATUS, k=1)[0],
        "bairro": bairro,
        "cidade": CIDADE,
        "localizacao": {"latitude": lat, "longitude": lon},
        "reportante": {"tipo": reportante_tipo, "identificador": identificador},
    }
    return evento


def gerar_dataset(quantidade: int, seed: int | None = None) -> list[dict]:
    '''Gera `quantidade` eventos sintéticos. Use `seed` para reprodutibilidade.'''
    if seed is not None:
        random.seed(seed)
    largura_id = max(8, len(str(quantidade)))
    eventos = [
        gerar_evento(i, largura_id=largura_id)
        for i in tqdm(range(1, quantidade + 1), desc="Gerando eventos")
    ]
    return eventos

## 5. Seleção interativa da quantidade de dados

Escolha um dos volumes sugeridos pelo enunciado (Pequeno = 1.000, Médio = 50.000,
Grande = 100.000) ou selecione **Personalizado** para digitar qualquer quantidade.
Depois clique em **Gerar Dados**.

In [ ]:
volume_dropdown = widgets.Dropdown(
    options=[
        ("Pequeno (1.000 registros)", 1000),
        ("Médio (50.000 registros)", 50000),
        ("Grande (100.000 registros)", 100000),
        ("Personalizado", "custom"),
    ],
    value=1000,
    description="Volume:",
    style={"description_width": "initial"},
)

quantidade_custom = widgets.BoundedIntText(
    value=1000,
    min=1,
    max=5_000_000,
    step=1000,
    description="Quantidade:",
    style={"description_width": "initial"},
    disabled=True,
)

botao_gerar = widgets.Button(description="Gerar dados", button_style="success", icon="database")
saida = widgets.Output()

df_eventos = pd.DataFrame()  # ficará populado após a geração


def _on_volume_change(change):
    quantidade_custom.disabled = change["new"] != "custom"


def _on_gerar_clicked(_):
    global df_eventos
    quantidade = (
        quantidade_custom.value if volume_dropdown.value == "custom" else volume_dropdown.value
    )
    with saida:
        clear_output(wait=True)
        print(f"Gerando {quantidade:,} eventos sintéticos...".replace(",", "."))
        eventos = gerar_dataset(quantidade, seed=SEED)
        df_eventos = pd.json_normalize(eventos)
        print(f"\nConcluído! {len(df_eventos):,} registros gerados.".replace(",", "."))
        display(df_eventos.head(10))


volume_dropdown.observe(_on_volume_change, names="value")
botao_gerar.on_click(_on_gerar_clicked)

display(widgets.HBox([volume_dropdown, quantidade_custom]), botao_gerar, saida)

## 6. Geração direta via código (alternativa sem widgets)

Se preferir não usar os controles interativos (por exemplo, ao rodar o notebook
via `nbconvert` / linha de comando), defina `QUANTIDADE` abaixo e execute esta
célula diretamente.

In [ ]:
QUANTIDADE = 1000  # troque para 50_000 ou 100_000 conforme o experimento desejado

if df_eventos.empty:
    eventos = gerar_dataset(QUANTIDADE, seed=SEED)
    df_eventos = pd.json_normalize(eventos)

print(f"Dataset atual em memória: {len(df_eventos):,} registros".replace(",", "."))
df_eventos.head()

## 7. Conferência rápida dos dados gerados

In [ ]:
print("Quantidade por tipo de evento:")
display(df_eventos["tipo"].value_counts())

print("\nQuantidade por bairro (top 10):")
display(df_eventos["bairro"].value_counts().head(10))

print("\nQuantidade por status:")
display(df_eventos["status"].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

df_eventos["tipo"].value_counts().plot(kind="barh", ax=axes[0])
axes[0].set_title("Eventos por tipo")
axes[0].invert_yaxis()

df_eventos["bairro"].value_counts().head(10).plot(kind="barh", ax=axes[1])
axes[1].set_title("Top 10 bairros com mais eventos")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Evolução temporal (quantidade de eventos por dia)
datas = pd.to_datetime(df_eventos["dataHora"]).dt.date
contagem_diaria = datas.value_counts().sort_index()

plt.figure(figsize=(13, 4))
contagem_diaria.plot()
plt.title("Evolução temporal — eventos por dia")
plt.xlabel("Data")
plt.ylabel("Quantidade de eventos")
plt.tight_layout()
plt.show()

In [ ]:
# Mapa rápido de dispersão geográfica (útil para validar a distribuição espacial
# antes de testar a consulta geográfica em raio)
plt.figure(figsize=(6, 7))
plt.scatter(
    df_eventos["localizacao.longitude"],
    df_eventos["localizacao.latitude"],
    s=4,
    alpha=0.4,
)
plt.title("Dispersão geográfica dos eventos gerados")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.tight_layout()
plt.show()

## 8. Exportação dos dados

Os arquivos são salvos na pasta `./dataset`, seguindo a estrutura de entregáveis
pedida no enunciado (`/scripts`, `/dataset`, `/docker`, `/documentacao`).

São gerados três formatos:

- **`eventos.json`** — array JSON único (útil para `mongoimport --jsonArray` ou para inspeção manual).
- **`eventos.jsonl`** — um objeto JSON por linha (formato *JSON Lines*, ideal para `mongoimport` sem a flag `--jsonArray`, e também compatível com ferramentas de carga do Cassandra/Redis via scripts próprios).
- **`eventos.csv`** — formato tabular, útil para inspeção rápida em planilhas ou para bancos que preferem carga via CSV.

In [ ]:
import os

PASTA_SAIDA = "./dataset"
os.makedirs(PASTA_SAIDA, exist_ok=True)

eventos_dict = df_eventos.copy()

# Reconstrói a estrutura aninhada original (localizacao / reportante) a partir
# das colunas "achatadas" pelo pandas, garantindo fidelidade ao esquema do enunciado.
def _linha_para_evento(row):
    return {
        "idEvento": row["idEvento"],
        "tipo": row["tipo"],
        "descricao": row["descricao"],
        "dataHora": row["dataHora"],
        "gravidade": int(row["gravidade"]),
        "status": row["status"],
        "bairro": row["bairro"],
        "cidade": row["cidade"],
        "localizacao": {
            "latitude": row["localizacao.latitude"],
            "longitude": row["localizacao.longitude"],
        },
        "reportante": {
            "tipo": row["reportante.tipo"],
            "identificador": row["reportante.identificador"],
        },
    }

eventos_finais = [_linha_para_evento(r) for _, r in eventos_dict.iterrows()]

# 1) JSON (array único)
caminho_json = os.path.join(PASTA_SAIDA, "eventos.json")
with open(caminho_json, "w", encoding="utf-8") as f:
    json.dump(eventos_finais, f, ensure_ascii=False, indent=2)

# 2) JSON Lines
caminho_jsonl = os.path.join(PASTA_SAIDA, "eventos.jsonl")
with open(caminho_jsonl, "w", encoding="utf-8") as f:
    for evento in eventos_finais:
        f.write(json.dumps(evento, ensure_ascii=False))
        f.write("\n")

# 3) CSV
caminho_csv = os.path.join(PASTA_SAIDA, "eventos.csv")
df_eventos.to_csv(caminho_csv, index=False, encoding="utf-8")

print("Arquivos exportados:")
print(f" - {caminho_json}  ({len(eventos_finais):,} registros)".replace(",", "."))
print(f" - {caminho_jsonl}")
print(f" - {caminho_csv}")

### Importando no MongoDB (exemplo)

```bash
# JSON Lines (recomendado)
mongoimport --db eventos_urbanos --collection eventos --file dataset/eventos.jsonl

# JSON como array único
mongoimport --db eventos_urbanos --collection eventos --file dataset/eventos.json --jsonArray
```

Para Cassandra ou Redis, use o `eventos.csv` ou `eventos.jsonl` como entrada de um
script de carga próprio (`COPY` no `cqlsh` para Cassandra, ou um script Python com
`redis-py` fazendo `SET`/`HSET`/`JSON.SET` para Redis).